This notebook tries to boost the output of the NNLS by fitting the residuals received from the NNLS by a MLP. This is an example of 'Ensemble Learning'. Right now, the NNLS alone performs better than this NNLS+MLP (so it does NOT WORK...)

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from scipy.optimize import nnls
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error
rng = np.random.default_rng(seed=None)
sys.path.insert(0, os.path.abspath(".."))
from build_spectra.linear_combination import generate_mixture_spectra

In [ ]:
df = pd.read_excel('../../../data/spectral_library_with_scattering.xlsx') # read the 'pure spectra' library with scattering
df.ffill(axis=0, inplace=True)
df.bfill(axis=0, inplace=True)  # handles leading NaNs
wavelength = df['Wavelength']

# this array serves as the 'basis function' for the NNLS 
abs_spectra = df.iloc[:, 1:].to_numpy().T # (n_wavelengths, n_species)

mixture, weights = generate_mixture_spectra(df, combination_sizes=(5,6,  7, 8, 9), n_mixtures_per_combination=100)
mixture, weights = mixture.to_numpy(), weights.to_numpy()
# need to cut out pure spectra and wavelength columns
mixture, weights = mixture[:,10:], weights[:,1:]

In [ ]:
noise = rng.normal(0, 1, size = mixture.shape)
noise_level = 0.1
# add 10% noise
mixture_noisy = mixture + noise_level * noise

lam = 10 * np.max(np.abs(mixture_noisy))  # relative to your signal scale
A = abs_spectra.T                                                    
A_aug = np.vstack([A, lam * np.ones((1, A.shape[1]))])               
b_aug = np.vstack([mixture_noisy, lam * np.ones((1, mixture_noisy.shape[1]))]) 


In [ ]:
# NNLS - scipy algorithm
# can give a warning: 'let NumPy figure it out'
estimated_conc = np.array([nnls(A_aug, b_aug[:, i])[0] for i in range(b_aug.shape[1])])

In [ ]:
X = torch.tensor(mixture_noisy.T, dtype=torch.float32) 
y = torch.tensor(weights.astype(np.float64), dtype=torch.float64)

# find residuals of estimated concentration received from NNLS
residuals = torch.tensor(estimated_conc - weights.astype(np.float64), dtype=torch.float32)

# shuffle
perm = torch.randperm(X.shape[0])    # shuffle along sample dimension
X, y, residuals = X[perm], y[perm], residuals[perm]

# split in training and validation
N_tot = X.shape[0]
N_training = int(N_tot * 0.8)

X_train, y_train = X[:N_training], y[:N_training]
X_val,   y_val   = X[N_training:], y[N_training:]
res_train = residuals[:N_training]
res_val   = residuals[perm[N_training:]]

# normalize data
mean = X_train.mean(dim=0, keepdim=True) 
std  = X_train.std(dim=0,  keepdim=True) + 1e-8

X_train_norm = (X_train - mean) / std
X_val_norm   = (X_val   - mean) / std

# transform estimated concentration into torch tensor
estimated_conc_t = torch.tensor(estimated_conc, dtype=torch.float32)
nnls_train = estimated_conc_t[perm[:N_training]]   # (N_training, 9)
nnls_val   = estimated_conc_t[perm[N_training:]]

# final training and validation data
X_mlp_train = torch.cat([X_train_norm, nnls_train], dim=1)  # (N_training, 451+9)
X_mlp_val   = torch.cat([X_val_norm,   nnls_val], dim=1)

In [ ]:
def training_loop_residual(nr_epochs, epochs_no_improvement, X_train_, res_train_, X_val_, res_val_, wavelength, N_species):
    X_train_ = X_train_.float()
    X_val_   = X_val_.float()
    res_train_ = res_train_.float()
    res_val_   = res_val_.float()

    # Input is spectrum + NNLS prediction concatenated
    input_size = len(wavelength) + N_species

    model = nn.Sequential(
        nn.Linear(input_size, 64),
        nn.ReLU(),
        nn.Linear(64, N_species),
        # No Sparsemax — residuals can be negative
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5, min_lr=1e-6)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses   = []
    train_maes   = []
    val_maes     = []
    epochs_list  = []

    X_train_   = X_train_.reshape(-1, input_size).float()
    X_val_     = X_val_.reshape(-1, input_size).float()
    res_train_ = res_train_.reshape(-1, N_species).float()
    res_val_   = res_val_.reshape(-1, N_species).float()

    best_val_loss = float('inf')
    epochs_without_improvement = 0

    for epoch in range(nr_epochs):
        model.train()
        optimizer.zero_grad()

        pred = model(X_train_)
        loss = criterion(pred, res_train_)   # target is residuals, not weights
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            train_mae_per_species = torch.mean(torch.abs(pred - res_train_), dim=0).tolist()

        model.eval()
        with torch.no_grad():
            pred_val = model(X_val_)
            loss_val = criterion(pred_val, res_val_)
            val_mae_per_species = torch.mean(torch.abs(pred_val - res_val_), dim=0).tolist()

        train_losses.append(loss.item())
        val_losses.append(loss_val.item())
        train_maes.append(train_mae_per_species)
        val_maes.append(val_mae_per_species)

        if loss_val.item() < best_val_loss:
            best_val_loss = loss_val.item()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement > epochs_no_improvement:
            print(f"Early stopping triggered at epoch {epoch}")
            break

        scheduler.step(loss_val.item())

    return train_losses, val_losses, train_maes, val_maes, epochs_list, model

In [ ]:
train_losses, val_losses, train_maes, val_maes, epochs_list, trained_model = training_loop_residual(10000, 8, X_mlp_train, res_train, X_mlp_val, res_val, wavelength, 9)

In [ ]:
def predict_residual(model, X_norm, nnls_pred):
    """
    Combine NNLS + MLP correction at inference time.
    X_norm:    normalized spectrum  (N, 451)
    nnls_pred: NNLS output          (N, 9)
    """
    model.eval()
    with torch.no_grad():
        X_input  = torch.cat([X_norm, nnls_pred], dim=1).float()  # (N, 460)
        correction = model(X_input)                                 # (N, 9)
        y_final  = nnls_pred + correction
    return y_final

In [ ]:
# get final validation weights (NNLS + residual MLP)
y_final_val = predict_residual(trained_model, X_val_norm, nnls_val)

# compare all three
y_val_np       = y_val.numpy()
nnls_val_np    = nnls_val.numpy()
y_final_val_np = y_final_val.numpy()

# per-species MAE
mae_nnls     = np.mean(np.abs(nnls_val_np    - y_val_np), axis=0)
mae_ensemble = np.mean(np.abs(y_final_val_np - y_val_np), axis=0)

print("Species        | NNLS MAE | Ensemble MAE")
print("-" * 45)
for i, (m1, m2) in enumerate(zip(mae_nnls, mae_ensemble)):
    better = "✓" if m2 < m1 else "✗"
    print(f"Species {i+1:>2}     |  {m1:.4f}  |  {m2:.4f}   {better}")

print(f"\nOverall NNLS MAE:     {mae_nnls.mean():.4f}")
print(f"Overall Ensemble MAE: {mae_ensemble.mean():.4f}")

From here on there are some plots that show results. Shapes of matrices can change depending on number of samples.

In [ ]:
# ground_truth shape: (4600, 9), estimated_conc shape: (4600, 9)
# Identify dominant species per mixture (highest true weight)
dominant_species = np.argmax(weights, axis=1)  # (4600,)

n_species = weights.shape[1]
heatmap = np.zeros((n_species, n_species))
species_names = df.columns[1:].tolist()

for i in range(n_species):
    # Select all mixtures where species i is dominant
    mask = dominant_species == i
    if mask.sum() > 0:
        # Average predicted concentration of each species j in those mixtures
        heatmap[i, :] = estimated_conc[mask].mean(axis=0)

# Optional: normalize each row so values sum to 1
heatmap_norm = heatmap / heatmap.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heatmap_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=species_names,
    yticklabels=species_names,
    ax=ax,
    vmin=0, vmax=1
)
ax.set_xlabel("Predicted species (mean estimated concentration)")
ax.set_ylabel("Dominant true species")
ax.set_title("NNLS: Mean predicted concentration\ngrouped by dominant true species")
plt.tight_layout()
plt.show()
# this does not show how the model performs. This plot picks dominant species. Dominant species can have weights of .31, so the spectra is still dominated by the other species! 
# That's why this does not show high diagonal values. To fix this, use a mask with a threshold for the weights -- only dominant species with weights > .6-.8 will show. 


In [ ]:
threshold = 0.5  # adjust as needed

# Print sample counts per species at each threshold
print("Sample counts per species at different thresholds:")
for thresh in [0.4, 0.5, 0.6, 0.7]:
    mask = weights.max(axis=1) > thresh
    counts = np.bincount(np.argmax(weights[mask], axis=1), minlength=9)
    print(f"  thresh={thresh}: {counts} (total={mask.sum()})")
print(weights[:5])
# Apply threshold: only keep mixtures where dominant species > threshold
mask_dominant = weights.max(axis=1) > threshold
gt_filtered   = weights[mask_dominant]
ec_filtered   = estimated_conc[mask_dominant]

dominant_species = np.argmax(gt_filtered, axis=1)

n_species = weights.shape[1]
heatmap = np.zeros((n_species, n_species))

for i in range(n_species):
    mask = dominant_species == i
    if mask.sum() > 0:
        heatmap[i, :] = ec_filtered[mask].mean(axis=0)

# Normalize rows to sum to 1
heatmap_norm = heatmap / heatmap.sum(axis=1, keepdims=True)

species_names = [
    "Diatom_Ptricornutum", "Diatom_Csimplex", "Chlamydomonas_Cpriscuii",
    "Chlamydomonas_Creindhardtii", "Dinoflagellate_Symbiodiniumsp",
    "Dinoflagellate_Smicroadriaticum", "Dinoflagellate_Dtrenchii",
    "Dinoflagellate_Cgoreaui", "Cyanobacteria_Synechosystis"
]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    heatmap_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=species_names,
    yticklabels=species_names,
    ax=ax,
    vmin=0, vmax=1
)
ax.set_xlabel("Predicted species (mean estimated concentration)")
ax.set_ylabel("Dominant true species")
ax.set_title(f"NNLS: Mean predicted concentration\ngrouped by dominant true species (threshold={threshold})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
# contuining from last comment, this works waaay better!

In [ ]:
n_species = len(species_names)
ground_truth = weights
# --- Per-species metrics ---
mae_per_species = [mean_absolute_error(ground_truth[:, i], estimated_conc[:, i]) for i in range(n_species)]
r2_per_species  = [r2_score(ground_truth[:, i], estimated_conc[:, i]) for i in range(n_species)]

# --- Scatter plots: true vs predicted per species ---
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()
print(r2_per_species)
for i, (name, ax) in enumerate(zip(species_names, axes)):
    ax.scatter(ground_truth[:, i], estimated_conc[:, i], alpha=0.1, s=5, color="steelblue")
    # Perfect prediction line
    lim = [0, max(ground_truth[:, i].max(), estimated_conc[:, i].max())]
    ax.plot(lim, lim, "r--", linewidth=1)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel("True", fontsize=7)
    ax.set_ylabel("Predicted", fontsize=7)
    ax.text(0.05, 0.92, f"MAE={mae_per_species[i]:.3f}\nR²={r2_per_species[i]:.3f}",
            transform=ax.transAxes, fontsize=7, verticalalignment='top')

plt.suptitle("NNLS: True vs Predicted concentration per species", fontsize=12)
plt.tight_layout()
plt.show()
# shows predicted concentration values on y-axis versus the true concentration on x-axis. If it is perfect, it should follow the red line (linear). 
# R^2 determines how well the linear fit is. For noise_level = 0, it is perfect!